# Wykonanie metryk (IoU) względem Ground Truth

In [121]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# Dodanie dźwięku przy długich komórkach
import subprocess
def play_sound():
    sound_file = '/mnt/d/Backup/INZ/msg.ogg'
    subprocess.run(['ffplay', '-nodisp', '-autoexit', sound_file], capture_output=True)

## Przygotowanie i obróbka danych

### Pobranie danych

In [133]:
DATA_PATH = Path('/mnt/d/Backup/MAGISTERSKIE/outputs')

Ścieżki do pliku

In [134]:
df = pd.read_csv(DATA_PATH / 'GDSAM_2000/df.csv')
color_paths = df.loc[:,"color_path"]
color_paths_for_llms = df.loc[20:119,"color_path"]

Ground Truth

In [135]:
ground_truth_bbox = np.load(DATA_PATH / 'GT/bboxes_normalized_2000.npy')
print(ground_truth_bbox.shape)

(2012, 12)


GroundedSAM

In [136]:
GroundedSAM_bbox = np.load(DATA_PATH / 'GDSAM_2000/all_boxes.npy')
print(GroundedSAM_bbox.shape)

(2012, 3, 4)


OpenAI

In [137]:
gpt_bbox = np.load(DATA_PATH / 'LLM/xy_array_gpt100.npy')
print(gpt_bbox.shape)

(100,)


Dostrojone OpenAI

In [138]:
gpt_ft_bbox = np.load(DATA_PATH / 'LLM/xy_array_gpt_ft.npy')
print(gpt_ft_bbox.shape)
# for i in range(0,10):print(f"ID {i}: {gpt_bbox[i]}\n")

(100,)


### Dopasowanie wszystkich do tego samego formatu

Widok wszystkich po pobraniu

In [139]:
i = 12
print(f"ground_truth_bbox: {ground_truth_bbox[i]}\n")
print(f"GroundedSAM_bbox: {GroundedSAM_bbox[i]}\n")
print(f"gpt_bbox: {gpt_bbox[i]}\n")
print(f"gpt_ft_bbox: {gpt_ft_bbox[i]}")

ground_truth_bbox: [0.31054688 0.2995283  0.35742188 0.3773585  0.35742188 0.25707546
 0.41992188 0.30660376 0.4375     0.3301887  0.45703125 0.41509435]

GroundedSAM_bbox: [[0.3013275  0.29014584 0.36623523 0.38551801]
 [0.35192299 0.25033733 0.42847854 0.31480581]
 [0.40702486 0.3132239  0.47420478 0.42812437]]

gpt_bbox: (0.285000,0.360000,0.335000,0.520000),(0.318000,0.300000,0.345000,0.360000),(0.365000,0.360000,0.445000,0.520000)

gpt_ft_bbox: (0.238281,0.396226,0.259766,0.452830),(0.271484,0.424528,0.300781,0.452830),(0.275391,0.452830,0.314453,0.490566)


Dopasowywanie OpenAI

In [140]:
def parse_bbox_rows(arr):
  out = []
  for row in arr:
    text = row[0] if isinstance(row, (list, np.ndarray)) else str(row)
    boxes = re.findall(r'\(([0-9]*\.?[0-9]+)\s*,\s*([0-9]*\.?[0-9]+)\s*,\s*([0-9]*\.?[0-9]+)\s*,\s*([0-9]*\.?[0-9]+)\)', text)
    nums = [float(x) for b in boxes for x in b]
    # dociecie/padding do 12 liczb (3 boxy po 4)
    if len(nums) < 12:
      nums += [0.0] * (12 - len(nums))
    else:
      nums = nums[:12]
    out.append(nums)
  return np.array(out, dtype=float)

gpt_bbox = parse_bbox_rows(gpt_bbox)
gpt_ft_bbox = parse_bbox_rows(gpt_ft_bbox)

print(f"OpenAI bbox shape: {gpt_bbox.shape}")
print(f"Przykład OpenAI bbox (pierwszy wiersz): \n{gpt_bbox[0]}\n")

print(f"Dostrojony OpenAI bbox shape: {gpt_ft_bbox.shape}")
print(f"Przykład dostrojonego OpenAI bbox (pierwszy wiersz): \n{gpt_ft_bbox[0]}")

OpenAI bbox shape: (100, 12)
Przykład OpenAI bbox (pierwszy wiersz): 
[0.4   0.38  0.46  0.6   0.354 0.3   0.392 0.38  0.47  0.42  0.54  0.62 ]

Dostrojony OpenAI bbox shape: (100, 12)
Przykład dostrojonego OpenAI bbox (pierwszy wiersz): 
[0.289062 0.396226 0.328125 0.481132 0.345703 0.396226 0.388672 0.433962
 0.353516 0.45283  0.396484 0.509434]


Dopasowywanie GroundedSAM

In [141]:
GroundedSAM_bbox = GroundedSAM_bbox.reshape(-1,12)

Stworzenie GDSAM i ground truth dla 100 obrazów testowych

In [142]:
GroundedSAM_bbox_100 = GroundedSAM_bbox[20:120]
ground_truth_bbox_100 = ground_truth_bbox[20:120]

Wyświetlenie wyników dopasowywania

In [143]:
i = 12
print(f"ground_truth_bbox: {ground_truth_bbox.shape} -> {ground_truth_bbox[i]}\n")
print(f"ground_truth_bbox_100: {ground_truth_bbox_100.shape} -> {ground_truth_bbox_100[i]}\n")
print(f"GroundedSAM_bbox: {GroundedSAM_bbox.shape} -> {GroundedSAM_bbox[i]}\n")
print(f"GroundedSAM_bbox_100: {GroundedSAM_bbox_100.shape} -> {GroundedSAM_bbox_100[i]}\n")
print(f"gpt_bbox: {gpt_bbox.shape} -> {gpt_bbox[i]}\n")
print(f"gpt_ft_bbox: {gpt_ft_bbox.shape} -> {gpt_ft_bbox[i]}")

ground_truth_bbox: (2012, 12) -> [0.31054688 0.2995283  0.35742188 0.3773585  0.35742188 0.25707546
 0.41992188 0.30660376 0.4375     0.3301887  0.45703125 0.41509435]

ground_truth_bbox_100: (100, 12) -> [0.23242188 0.4292453  0.26171875 0.4787736  0.2578125  0.41509435
 0.28515625 0.46698114 0.26171875 0.48349056 0.3203125  0.5283019 ]

GroundedSAM_bbox: (2012, 12) -> [0.3013275  0.29014584 0.36623523 0.38551801 0.35192299 0.25033733
 0.42847854 0.31480581 0.40702486 0.3132239  0.47420478 0.42812437]

GroundedSAM_bbox_100: (100, 12) -> [0.22168809 0.41595042 0.26852518 0.49022976 0.24849398 0.40516564
 0.29193011 0.47497877 0.2501969  0.46269608 0.32457122 0.54319108]

gpt_bbox: (100, 12) -> [0.285 0.36  0.335 0.52  0.318 0.3   0.345 0.36  0.365 0.36  0.445 0.52 ]

gpt_ft_bbox: (100, 12) -> [0.238281 0.396226 0.259766 0.45283  0.271484 0.424528 0.300781 0.45283
 0.275391 0.45283  0.314453 0.490566]


## IoU

In [144]:
i = 12
print(f"ground_truth_bbox: {ground_truth_bbox.shape} -> {ground_truth_bbox[i]}\n")
print(f"ground_truth_bbox_100: {ground_truth_bbox_100.shape} -> {ground_truth_bbox_100[i]}\n")
print(f"GroundedSAM_bbox: {GroundedSAM_bbox.shape} -> {GroundedSAM_bbox[i]}\n")
print(f"GroundedSAM_bbox_100: {GroundedSAM_bbox_100.shape} -> {GroundedSAM_bbox_100[i]}\n")
print(f"gpt_bbox: {gpt_bbox.shape} -> {gpt_bbox[i]}\n")
print(f"gpt_ft_bbox: {gpt_ft_bbox.shape} -> {gpt_ft_bbox[i]}")

ground_truth_bbox: (2012, 12) -> [0.31054688 0.2995283  0.35742188 0.3773585  0.35742188 0.25707546
 0.41992188 0.30660376 0.4375     0.3301887  0.45703125 0.41509435]

ground_truth_bbox_100: (100, 12) -> [0.23242188 0.4292453  0.26171875 0.4787736  0.2578125  0.41509435
 0.28515625 0.46698114 0.26171875 0.48349056 0.3203125  0.5283019 ]

GroundedSAM_bbox: (2012, 12) -> [0.3013275  0.29014584 0.36623523 0.38551801 0.35192299 0.25033733
 0.42847854 0.31480581 0.40702486 0.3132239  0.47420478 0.42812437]

GroundedSAM_bbox_100: (100, 12) -> [0.22168809 0.41595042 0.26852518 0.49022976 0.24849398 0.40516564
 0.29193011 0.47497877 0.2501969  0.46269608 0.32457122 0.54319108]

gpt_bbox: (100, 12) -> [0.285 0.36  0.335 0.52  0.318 0.3   0.345 0.36  0.365 0.36  0.445 0.52 ]

gpt_ft_bbox: (100, 12) -> [0.238281 0.396226 0.259766 0.45283  0.271484 0.424528 0.300781 0.45283
 0.275391 0.45283  0.314453 0.490566]


Wykonanie metryki IoU dla GroundedSAM i 2000 plików

In [153]:
def box_iou(box_a, box_b):
    """IoU dla dwóch boxów w formacie [x1, y1, x2, y2]."""
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter_area

    return 0.0 if union <= 0 else inter_area / union


def image_iou(gt_row, pred_row):
    """Średnie IoU dla jednego obrazu (boxy parowane po indeksie)."""
    gt_boxes = np.asarray(gt_row, dtype=float).reshape(-1, 4)
    pred_boxes = np.asarray(pred_row, dtype=float).reshape(-1, 4)

    valid_mask = np.array([
        (b[2] > b[0]) and (b[3] > b[1]) and np.any(b != 0.0)
        for b in gt_boxes
    ])

    if not valid_mask.any():
        return np.nan

    ious = [
        box_iou(gt_boxes[i], pred_boxes[i])
        for i in range(min(len(gt_boxes), len(pred_boxes)))
        if valid_mask[i]
    ]
    return float(np.mean(ious)) if ious else np.nan


def summarize_iou(iou_values, label):
    valid = iou_values[~np.isnan(iou_values)]
    print(f"\n--- {label} ---")
    print(f"Liczba próbek z poprawnym GT: {len(valid)}")
    if len(valid) == 0:
        print("Brak próbek do podsumowania.")
        return np.nan
    print(f"Średnie IoU: {valid.mean():.4f}")
    print(f"Mediana IoU: {np.median(valid):.4f}")
    print(f"Min/Max IoU: {valid.min():.4f} / {valid.max():.4f}")
    return valid.mean()


def to_excluded_indices(incomplete_detect, n):
    """Konwersja tablicy incomplete do zbioru indeksów 0-based do wykluczenia."""
    arr = np.asarray(incomplete_detect)

    if arr.dtype == bool:
        if arr.size != n:
            raise ValueError(
                f"Maska bool ma długość {arr.size}, a liczba próbek to {n}."
            )
        return set(np.where(arr)[0].tolist())

    flat = arr.reshape(-1)
    excluded = {int(x) for x in flat if np.isfinite(x)}
    excluded = {i for i in excluded if 0 <= i < n}
    return excluded


# IoU dla GroundedSAM na pełnym zbiorze
n = min(len(ground_truth_bbox), len(GroundedSAM_bbox))
iou_per_image_gdsam = np.array([
    image_iou(ground_truth_bbox[i], GroundedSAM_bbox[i])
    for i in range(n)
], dtype=float)

# Id z brakującą detekcją
incomplete_detect_GDSAM = np.load(DATA_PATH / 'GDSAM_2000/incomplete_detect.npy')
excluded_idx = to_excluded_indices(incomplete_detect_GDSAM, n)

# IoU po odfiltrowaniu incomplete id
keep_mask = np.array([i not in excluded_idx for i in range(n)], dtype=bool)
iou_per_image_gdsam_filtered = iou_per_image_gdsam[keep_mask]

print(f"Liczba wszystkich porównanych próbek: {n}")
print(f"Liczba id do wykluczenia (incomplete): {len(excluded_idx)}")
print(f"Liczba próbek po filtracji: {keep_mask.sum()}")

mean_full = summarize_iou(iou_per_image_gdsam, "GroundedSAM - pełny zbiór")
mean_filtered = summarize_iou(iou_per_image_gdsam_filtered, "GroundedSAM - bez incomplete id")

if np.isfinite(mean_full) and np.isfinite(mean_filtered):
    delta = mean_filtered - mean_full
    print(f"\nZmiana średniego IoU po filtracji: {delta:+.4f}")

# Podgląd pierwszych 10 IoU po filtracji
valid_filtered = iou_per_image_gdsam_filtered[~np.isnan(iou_per_image_gdsam_filtered)]
print("Pierwsze 10 IoU po filtracji:", np.round(valid_filtered[:10], 4))

Liczba wszystkich porównanych próbek: 2012
Liczba id do wykluczenia (incomplete): 291
Liczba próbek po filtracji: 1721

--- GroundedSAM - pełny zbiór ---
Liczba próbek z poprawnym GT: 1999
Średnie IoU: 0.4421
Mediana IoU: 0.4682
Min/Max IoU: 0.0000 / 0.6892

--- GroundedSAM - bez incomplete id ---
Liczba próbek z poprawnym GT: 1721
Średnie IoU: 0.4661
Mediana IoU: 0.4786
Min/Max IoU: 0.1308 / 0.6735

Zmiana średniego IoU po filtracji: +0.0240
Pierwsze 10 IoU po filtracji: [0.3635 0.4771 0.4597 0.4668 0.4485 0.4612 0.4898 0.5215 0.4889 0.5341]


In [ ]:
# IoU dla zbiorów testowych (100 próbek): GroundedSAM, OpenAI, OpenAI FT
TEST_START, TEST_END = 20, 120  # zakres zgodny z *_100
GT_100 = ground_truth_bbox_100

def evaluate_iou_100(label, gt_arr, pred_arr, excluded_local_idx=None):
    n_local = min(len(gt_arr), len(pred_arr))
    iou_values = np.array([
        image_iou(gt_arr[i], pred_arr[i])
        for i in range(n_local)
    ], dtype=float)

    print(f"\n===== {label} (n={n_local}) =====")
    mean_full = summarize_iou(iou_values, f"{label} - pełny zbiór")

    if excluded_local_idx is not None and len(excluded_local_idx) > 0:
        keep_mask = np.array([i not in excluded_local_idx for i in range(n_local)], dtype=bool)
        iou_filtered = iou_values[keep_mask]
        mean_filtered = summarize_iou(iou_filtered, f"{label} - bez incomplete id")
        if np.isfinite(mean_full) and np.isfinite(mean_filtered):
            print(f"Zmiana średniego IoU po filtracji: {mean_filtered - mean_full:+.4f}")
        return {
            "label": label,
            "n": n_local,
            "excluded": len(excluded_local_idx),
            "mean_full": mean_full,
            "mean_filtered": mean_filtered,
        }

    return {
        "label": label,
        "n": n_local,
        "excluded": 0,
        "mean_full": mean_full,
        "mean_filtered": np.nan,
    }


# incomplete dla GroundedSAM na zakresie 100 (globalne id -> lokalne 0..99)
incomplete_detect_GDSAM = np.load(DATA_PATH / 'GDSAM_2000/incomplete_detect.npy')
excluded_global = to_excluded_indices(incomplete_detect_GDSAM, len(ground_truth_bbox))
excluded_local_100 = {i - TEST_START for i in excluded_global if TEST_START <= i < TEST_END}

results_100 = [
    evaluate_iou_100(
        "GroundedSAM_100", GT_100, GroundedSAM_bbox_100, excluded_local_100
    )
]
results_100.append(evaluate_iou_100("OpenAI_100", GT_100, gpt_bbox))
results_100.append(evaluate_iou_100("OpenAI_FT_100", GT_100, gpt_ft_bbox))

summary_100 = pd.DataFrame(results_100)
print("\nPodsumowanie IoU (100 próbek):")
display(summary_100)


===== GroundedSAM_100 (n=100) =====

--- GroundedSAM_100 - pełny zbiór ---
Liczba próbek z poprawnym GT: 100
Średnie IoU: 0.4736
Mediana IoU: 0.4846
Min/Max IoU: 0.1786 / 0.5743

--- GroundedSAM_100 - bez incomplete id ---
Liczba próbek z poprawnym GT: 98
Średnie IoU: 0.4749
Mediana IoU: 0.4860
Min/Max IoU: 0.1786 / 0.5743
Zmiana średniego IoU po filtracji: +0.0012

===== OpenAI_100 (n=100) =====

--- OpenAI_100 - pełny zbiór ---
Liczba próbek z poprawnym GT: 100
Średnie IoU: 0.0131
Mediana IoU: 0.0000
Min/Max IoU: 0.0000 / 0.2698

===== OpenAI_FT_100 (n=100) =====

--- OpenAI_FT_100 - pełny zbiór ---
Liczba próbek z poprawnym GT: 100
Średnie IoU: 0.1292
Mediana IoU: 0.0895
Min/Max IoU: 0.0000 / 0.5507

Podsumowanie IoU (100 próbek):


,label,n,excluded,mean_full,mean_filtered
0,GroundedSAM_100,100,2,0.473609,0.474852
1,OpenAI_100,100,0,0.013055,NaN
2,OpenAI_FT_100,100,0,0.129153,NaN
